## Custom Task: The Recycling Robot MDP Implementation
This models the classic Sutton & Barto **Recycling Robot** paradigm. The agent collects soda cans with battery states: **High** or **Low**.

In [ ]:
import random

class RecyclingRobotEnv:
    """
    Recycling Robot Environment modeled on Sutton & Barto Example 3.2
    """
    def __init__(self, alpha=0.8, beta=0.7, r_search=2.0, r_wait=0.5):
        # MDP Spaces Configuration
        self.states = ["High", "Low"]
        self.actions = ["Search", "Wait", "Recharge"]

        # Parameter constants
        self.alpha = alpha       # Prob of keeping High charge during search
        self.beta = beta         # Prob of keeping Low charge during search
        self.r_search = r_search # Reward earned while actively searching
        self.r_wait = r_wait     # Passive collection reward

        self.reset()

    def reset(self):
        self.state = "High"
        return self.state

    def step(self, action):
        if action not in self.actions:
            raise ValueError(f"Invalid action: {action}")

        reward = 0.0
        done = False

        if self.state == "High":
            if action == "Search":
                reward = self.r_search
                # High -> High with probability alpha, else flips to Low
                self.state = "High" if random.random() < self.alpha else "Low"
            elif action == "Wait":
                reward = self.r_wait
                self.state = "High"
            elif action == "Recharge":
                # Invalid/Useless to recharge at high power
                reward = 0.0
                self.state = "High"

        elif self.state == "Low":
            if action == "Search":
                # Low -> Low with probability beta
                if random.random() < self.beta:
                    reward = self.r_search
                    self.state = "Low"
                else:
                    # Battery completely depleted, requires manual recovery rescue
                    reward = -3.0
                    self.state = "High"
                    done = True
            elif action == "Wait":
                reward = self.r_wait
                self.state = "Low"
            elif action == "Recharge":
                reward = 0.0
                self.state = "High"

        return self.state, reward, done

# --- Demonstration Running Simulated Traces ---
env = RecyclingRobotEnv()
state = env.reset()
print(f"Initial Robot State: {state}")

demo_actions = ["Search", "Search", "Wait", "Search", "Recharge", "Search"]
for step_num, act in enumerate(demo_actions, 1):
    next_state, reward, done = env.step(act)
    print(f"Step {step_num}: Current={state} | Action Chosen={act} -> Next={next_state} | Reward={reward}")
    state = next_state
    if done:
        print("Robot ran out of battery! Manual system rescue triggered.\n")
        break

Initial Robot State: High
Step 1: Current=High | Action Chosen=Search -> Next=High | Reward=2.0
Step 2: Current=High | Action Chosen=Search -> Next=Low | Reward=2.0
Step 3: Current=Low | Action Chosen=Wait -> Next=Low | Reward=0.5
Step 4: Current=Low | Action Chosen=Search -> Next=High | Reward=-3.0
Robot ran out of battery! Manual system rescue triggered.

